# 01 · Zero-shot Baseline — Qwen2.5-VL-3B (4-bit) trên ViVQA

**Mục tiêu:** đo baseline zero-shot (EM · VQA-Accuracy · ANLS) của Qwen2.5-VL-3B trên **tập test ViVQA (3.001 mẫu)** — con số tham chiếu để mọi cải tiến (QLoRA / OCR / RAG) so sánh với, dưới điều kiện y hệt.

**Nguyên tắc của notebook này:**
- Mọi tham số (model, `max_pixels`, dtype, prompt, metric) **đọc từ `configs/qwen_lora.yaml`** — không gõ tay.
- **Run all an toàn:** data có sẵn thì tự bỏ qua bước tải; phần chẩn đoán mặc định tắt; chạy full test theo **lô 500 mẫu, lưu lên Drive, tự resume** nếu Colab rớt.

**Điều kiện:** Runtime → Change runtime type → **T4 GPU**.

## 1. Thiết lập môi trường

Mount Drive, clone/pull code mới nhất từ GitHub, cài thư viện (pin `transformers==5.17.0` để tái lập), thêm repo vào `sys.path`.

*Kỳ vọng: in version + `✅ Thiết lập xong`. ~1 phút.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
sys.path.insert(0, '/content/ViVQA-VLM')

# Pin transformers = bản đã dùng ở W05 (tái lập). accelerate/bitsandbytes/qwen-vl-utils lấy bản mới.
!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes

import transformers, torch
print('transformers:', transformers.__version__, '| torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU -> đổi runtime sang T4')
print('✅ Thiết lập xong')

## 2. Chuẩn bị data ViVQA (tự bỏ qua nếu đã có)

Chỉ chạy thật khi Drive của account này **chưa có** data (vd account mới). Có đủ `train/val/test.json` + `images/` thì bỏ qua ngay. Seed 42 → split giống hệt mọi account.

*Kỳ vọng: `Data ViVQA đã có -> bỏ qua` (hoặc ~5 phút tải lần đầu).*

In [ ]:
VIVQA = '/content/drive/MyDrive/ViVQA-VLM/data/vivqa'
need = [os.path.join(VIVQA, f) for f in ('train.json', 'val.json', 'test.json', 'prep_report.json')]
if all(os.path.exists(p) for p in need) and os.path.isdir(os.path.join(VIVQA, 'images')):
    print('Data ViVQA đã có -> bỏ qua')
else:
    !python -m src.data.prepare_vivqa --out_dir "{VIVQA}" --val_ratio 0.1 --seed 42 --workers 16

## 3. Nạp model + processor THEO CONFIG

Đọc `configs/qwen_lora.yaml` (bản chốt W05: `max_pixels = 512*28*28`, dtype fp16 cho T4) và dựng model 4-bit NF4 + processor từ đúng các giá trị đó. Có `assert` chặn nếu config chưa phải bản W05. Ghi lại git commit để đưa vào log.

*Kỳ vọng: ~3–4 phút, in cấu hình + `✅ Model + processor sẵn sàng`.*

In [ ]:
import yaml, subprocess, copy
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

with open('/content/ViVQA-VLM/configs/qwen_lora.yaml', encoding='utf-8') as f:
    base_cfg = yaml.safe_load(f)
m, q = base_cfg['model'], base_cfg['quantization']
assert m['max_pixels'] == 401408 and m['torch_dtype'] == 'float16', 'Config chưa phải bản W05 -> chạy lại phần 1 (git pull)'

bnb = BitsAndBytesConfig(
    load_in_4bit=q['load_in_4bit'],
    bnb_4bit_quant_type=q['bnb_4bit_quant_type'],
    bnb_4bit_use_double_quant=q['bnb_4bit_use_double_quant'],
    bnb_4bit_compute_dtype=getattr(torch, q['bnb_4bit_compute_dtype']),
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    m['model_id'], quantization_config=bnb, device_map='auto',
    torch_dtype=getattr(torch, m['torch_dtype']),
)
processor = AutoProcessor.from_pretrained(
    m['model_id'], min_pixels=m['min_pixels'], max_pixels=m['max_pixels'],
)

GIT_COMMIT = subprocess.run(['git', '-C', '/content/ViVQA-VLM', 'rev-parse', '--short', 'HEAD'],
                            capture_output=True, text=True).stdout.strip()
print('model     :', m['model_id'])
print('pixels    :', m['min_pixels'], '->', m['max_pixels'])
print('dtype     :', m['torch_dtype'], '| compute:', q['bnb_4bit_compute_dtype'])
print('git commit:', GIT_COMMIT)
print('✅ Model + processor sẵn sàng')

## 4. Chép ảnh ViVQA từ Drive về đĩa Colab

Đọc ảnh lẻ trên Drive rất chậm → chép cả thư mục về `/content` một lần. Data gốc vẫn an toàn trên Drive.

*Kỳ vọng: ~1 phút, `Đã chép 10328 ảnh` (hoặc `Đã có sẵn`).*

In [ ]:
import shutil, time
SRC, LOCAL_IMAGES = os.path.join(VIVQA, 'images'), '/content/vivqa_images'
if not os.path.exists(LOCAL_IMAGES):
    t0 = time.time()
    shutil.copytree(SRC, LOCAL_IMAGES)
    print(f'Đã chép {len(os.listdir(LOCAL_IMAGES))} ảnh trong {time.time()-t0:.0f}s')
else:
    print('Đã có sẵn:', len(os.listdir(LOCAL_IMAGES)), 'ảnh')

## 5. (Tùy chọn) Chẩn đoán nhanh — 20 mẫu VAL

Chỉ dùng khi muốn soi nhanh PRED vs GOLD sau khi đổi prompt/metric. **Mặc định tắt** (`RUN_DIAG = False`) để Run all không tốn thời gian. Dùng tập VAL, không đụng TEST.

*Kỳ vọng khi bật: ~20 giây, in metrics + 20 ví dụ.*

In [ ]:
RUN_DIAG = False   # đổi thành True nếu muốn chẩn đoán

import json
from src.data.vivqa_dataset import load_vivqa
from src.eval.run_eval import run_evaluation

if RUN_DIAG:
    diag = load_vivqa(os.path.join(VIVQA, 'val.json'), image_dir=LOCAL_IMAGES, max_samples=20)
    dcfg = copy.deepcopy(base_cfg)
    dcfg['run']['name'] = 'W05_zeroshot_qwen25vl_vivqa_val20_diag'
    dcfg['data']['test_path'] = os.path.join(VIVQA, 'val.json')
    r = run_evaluation(model, processor, diag, dcfg, verbose_every=5)
    print(json.dumps(r['metrics'], ensure_ascii=False, indent=2))
    for x in r['predictions']:
        print(f"Q: {x['question']}\n   PRED: {x['prediction']!r}\n   GOLD: {x['answers']}\n")
else:
    print('Bỏ qua chẩn đoán (RUN_DIAG = False)')

## 6. Chạy FULL TEST theo lô (resume được)

Chia 3.001 mẫu test thành các lô 500. Mỗi lô xong được **ghi ngay lên Drive** (`experiments/<RUN>/chunks/`). Nếu Colab rớt: chạy lại phần 1→4 rồi chạy lại cell này — **lô nào có rồi tự bỏ qua**. File được ghi qua `.tmp` rồi mới đổi tên, nên lô dở dang không bị tính là xong.

*Kỳ vọng: 7 lô, mỗi lô ~8–9 phút, tổng ~50 phút. Giữ tab Colab mở.*

In [ ]:
RUN_NAME  = 'W05_zeroshot_qwen25vl_vivqa_test_seed42'
OUT_DIR   = f'/content/drive/MyDrive/ViVQA-VLM/experiments/{RUN_NAME}'
CHUNK_DIR = os.path.join(OUT_DIR, 'chunks')
os.makedirs(CHUNK_DIR, exist_ok=True)
CHUNK = 500

test_samples = load_vivqa(os.path.join(VIVQA, 'test.json'), image_dir=LOCAL_IMAGES)
missing = [s['image_path'] for s in test_samples if not os.path.exists(s['image_path'])]
assert not missing, f'{len(missing)} ảnh thiếu, vd {missing[0]}'
print('Tổng mẫu test:', len(test_samples))

cfg = copy.deepcopy(base_cfg)
cfg['run']['name'] = RUN_NAME
cfg['data']['dataset_name'] = 'vivqa'
cfg['data']['test_path'] = os.path.join(VIVQA, 'test.json')
cfg['prompting']['mode'] = 'zero_shot'

n_chunks = (len(test_samples) + CHUNK - 1) // CHUNK
for k in range(n_chunks):
    path = os.path.join(CHUNK_DIR, f'chunk_{k:02d}.json')
    if os.path.exists(path):
        print(f'[lô {k+1}/{n_chunks}] đã có -> bỏ qua')
        continue
    part = test_samples[k * CHUNK:(k + 1) * CHUNK]
    res = run_evaluation(model, processor, part, cfg, verbose_every=100)
    with open(path + '.tmp', 'w', encoding='utf-8') as f:
        json.dump({'predictions': res['predictions'],
                   'inference_time_sec': res['meta']['inference_time_sec'],
                   'gpu': res['meta']['hardware']['gpu']}, f, ensure_ascii=False)
    os.replace(path + '.tmp', path)
    print(f"[lô {k+1}/{n_chunks}] xong {len(part)} mẫu | {res['meta']['inference_time_sec']:.0f}s | EM lô = {res['metrics']['exact_match']:.3f}")
print('✅ Đủ', n_chunks, 'lô')

## 7. Gộp kết quả + tính metric toàn tập + ghi log tái lập

Gộp mọi lô, kiểm tra đủ và không trùng mẫu, tính EM / VQA-Acc / ANLS trên **toàn bộ** test (kèm tách theo loại câu hỏi). Ghi `metrics.json` (kèm đủ meta: split, seed, GPU, version, config, git commit, thời gian) và `predictions.json` lên Drive.

*Kỳ vọng: in bảng metric + số mẫu từng loại. Đây là **con số baseline chính thức**.*

In [ ]:
import glob, platform
from collections import Counter
from datetime import datetime, timezone
from src.eval.metrics import evaluate_predictions

files = sorted(glob.glob(os.path.join(CHUNK_DIR, 'chunk_*.json')))
assert len(files) == n_chunks, f'Mới có {len(files)}/{n_chunks} lô -> chạy lại phần 6'
records, total_time, gpus = [], 0.0, set()
for fp in files:
    with open(fp, encoding='utf-8') as f:
        d = json.load(f)
    records += d['predictions']; total_time += d['inference_time_sec']; gpus.add(d['gpu'])
assert len(records) == len(test_samples), 'Số dự đoán khác số mẫu test'
assert len({r['question_id'] for r in records}) == len(records), 'Có mẫu bị trùng'

metrics = evaluate_predictions(
    [r['prediction'] for r in records], [r['answers'] for r in records],
    metric_names=cfg['eval']['primary_metrics'], anls_threshold=cfg['eval']['anls_threshold'],
    question_types=[r['question_type'] for r in records],
)
type_counts = Counter(r['question_type'] for r in records)

meta = {
    'run_name': RUN_NAME,
    'finished_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'dataset': 'vivqa', 'split': 'test.json', 'num_samples': len(records),
    'question_type_counts': dict(type_counts),
    'prompting_mode': 'zero_shot', 'decoding': 'greedy (do_sample=False)',
    'max_new_tokens': cfg['prompting'].get('max_new_tokens', 32),
    'model_id': m['model_id'],
    'quantization': {'4bit': True, 'type': q['bnb_4bit_quant_type'],
                     'double_quant': q['bnb_4bit_use_double_quant'],
                     'compute_dtype': q['bnb_4bit_compute_dtype']},
    'torch_dtype': m['torch_dtype'],
    'min_pixels': m['min_pixels'], 'max_pixels': m['max_pixels'],
    'seed': cfg['run']['seed'], 'git_commit': GIT_COMMIT,
    'config_file': 'configs/qwen_lora.yaml',
    'hardware': {'gpus_seen': sorted(gpus), 'platform': platform.platform()},
    'versions': {'transformers': transformers.__version__, 'torch': torch.__version__},
    'inference_time_sec': round(total_time, 1),
    'sec_per_sample': round(total_time / len(records), 3),
    'chunk_size': CHUNK,
}
with open(os.path.join(OUT_DIR, 'metrics.json'), 'w', encoding='utf-8') as f:
    json.dump({'metrics': metrics, 'meta': meta}, f, ensure_ascii=False, indent=2)
with open(os.path.join(OUT_DIR, 'predictions.json'), 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

if len(gpus) > 1:
    print('⚠️ Các lô chạy trên nhiều loại GPU khác nhau:', gpus, '(đã ghi vào meta)')
print('==== BASELINE CHÍNH THỨC — zero-shot, ViVQA test ====')
for k in ('exact_match', 'vqa_accuracy', 'anls'):
    print(f'{k:14s}: {metrics[k]:.4f}')
print('\n---- theo loại câu hỏi (số mẫu) ----')
for t, c in type_counts.most_common():
    print(f"{t:10s} n={c:5d} | EM={metrics[f'exact_match__{t}']:.3f} | ANLS={metrics[f'anls__{t}']:.3f}")
print(f"\nThời gian: {meta['inference_time_sec']}s ({meta['sec_per_sample']} s/mẫu) | GPU: {sorted(gpus)} | commit {GIT_COMMIT}")
print('Đã lưu ->', OUT_DIR)